In [0]:
# Here we are going to make Fct tables and dimension tabls 
# Understanding the joins
# orders      -> Order information
# customers   -> Customer attributes
# order_items -> Product sold + Seller + Price
# products    -> Product details
# category_translation/product_category_name -> English category names
# payments    -> Payment amount

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
orders = spark.table("silver.orders")
customers = spark.table("silver.customers")
items = spark.table("silver.order_items")
products = spark.table("silver.products")
payments = spark.table("silver.payments")
categories = spark.table("silver.category_translation")
sellers=spark.table("silver.sellers")

In [0]:
# FCT_Sales
fct_sales = (
    items.alias("i")

    .join(
        orders.alias("o"),
        "order_id",
        "inner"
    )

    .join(
        customers.alias("c"),
        "customer_id",
        "inner"
    )

    .join(
        products.alias("p"),
        "product_id",
        "left"
    )

    .join(
        sellers.alias("s"),
        "seller_id",
        "left"
    )

    .select(
        # Keys
        "order_id",
        "customer_id",
        "product_id",
        "seller_id",

        # Customer
        "customer_city",
        "customer_state",

        # Seller
        "seller_city",
        "seller_state",

        # Product
        "product_category_name",
        "product_weight_g",
        "product_volume_cm3",

        # Dates
        "order_date",
        "order_year",
        "order_month",
        "order_month_name",

        # Order
        "order_status",

        # Metrics
        "price",
        "freight_value",
        "total_item_value"
    )
)


In [0]:
# fct_sales.printSchema()
# fct_sales.show(5)
# fct_sales.count()

# Revenue Validation
from pyspark.sql import functions as F
fct_sales.select(
    F.sum("price").alias("product_revenue"),
    F.sum("freight_value").alias("freight_revenue"),
    F.sum("total_item_value").alias("gross_revenue")
).show()

In [0]:
(
    fct_sales.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.fct_sales")
)

spark.sql("""
SELECT COUNT(*)
FROM gold.fct_sales
""").show()

KPI 1 - MONTHLY REVENUE


In [0]:
monthly_revenue = (
    fct_sales
    .groupBy(
        "order_year",
        "order_month"
    )
    .agg(
        F.round(
            F.sum("total_item_value"),
            2
        ).alias("revenue")
    )
    .orderBy(
        "order_year",
        "order_month"
    )
)
monthly_revenue.show()
monthly_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.kpi_monthly_revenue")

KPI 2 — Revenue by State


In [0]:
state_sales = (
    fct_sales
    .groupBy("customer_state")
    .agg(
        F.round(
            F.sum("total_item_value"),
            2
        ).alias("revenue")
    )
    .orderBy(
        F.desc("revenue")
    )
)
state_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.kpi_state_sales")

KPI 3 — Revenue by Product Category

In [0]:
category_sales = (
    fct_sales
    .groupBy("product_category_name")
    .agg(
        F.round(
            F.sum("total_item_value"),
            2
        ).alias("revenue")
    )
    .orderBy(
        F.desc("revenue")
    )
)
category_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.kpi_category_sales")

KPI 4 — Top 20 Products

In [0]:
top_products = (
    fct_sales
    .groupBy("product_id")
    .agg(
        F.round(
            F.sum("total_item_value"),
            2
        ).alias("revenue")
    )
    .orderBy(
        F.desc("revenue")
    )
)
top_products.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.kpi_top_products")

In [0]:
%sql
SHOW TABLES IN gold;